# DEKM_F paper-text diagnostic: RCV1 seed 42

This is the corrected bounded diagnostic after separating the paper RMS normalization from the released unit-L2 loader. It is not a full reproduction claim. It runs only RCV1 seed 42 with the settings stated in the DEKM 2021 paper text: Xavier initialization, symmetric eigendecomposition, full-dataset DEKM_F refinement, and a 0.1% assignment-change tolerance. Deterministic seeding is added so the comparison can be audited. The run is compared with the released-source DEKM_F artifact and the paper reference before deciding whether more seeds are justified.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a GPU accelerator in Kaggle notebook settings."


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/kaggle/working/mvdec_fpmax")
subprocess.run(["rm", "-rf", str(REPO)], check=True)
subprocess.run([
    "git", "clone", "--branch", "dev", "--single-branch",
    "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO),
], check=True)
EXPECTED_REVISION = "d2a39fcd312b27c00866fa8cd6a84300f3c18b16"
subprocess.run(
    ["git", "checkout", "--detach", EXPECTED_REVISION], cwd=REPO, check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"],
    check=True,
)
revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
assert revision == EXPECTED_REVISION, revision
print("Repository revision:", revision)

subprocess.run([
    sys.executable, str(REPO / "scripts/fetch_dekm_release.py"),
    "--datasets", "RCV1",
], cwd=REPO, check=True)
from sklearn.datasets import fetch_rcv1
fetch_rcv1(download_if_missing=True)
print("RCV1 source is ready.")


In [ ]:
import json

SEED = 42
PROTOCOL = "dekm_f_2021_paper_text_seeded_modern_runtime_v1"
REPORT_ROOT = Path("/kaggle/working/dekm-f-paper-text-rcv1-seed42")
RUN_ROOT = REPORT_ROOT / "runs"
command = [
    sys.executable,
    "-m",
    "representation_learning.DEKM_dense",
    "RCV1",
    "--protocol",
    "paper-text-full-batch-2021",
    "--seed",
    str(SEED),
    "--runs",
    "1",
    "--require-gpu",
    "--dataset-root",
    str(REPO / "external_repos/DEKM/datasets"),
    "--output-dir",
    str(RUN_ROOT),
]
subprocess.run(command, cwd=REPO, check=True)
print("Training complete.")


In [ ]:
import pandas as pd

PAPER = {"acc": 0.6212, "nmi": 0.3598}
RELEASED_SOURCE_SEED42 = {"acc": 0.7404, "nmi": 0.5058763446541975}
protocol_root = RUN_ROOT / "rcv1" / PROTOCOL
config_roots = [path for path in protocol_root.iterdir() if path.is_dir()]
assert len(config_roots) == 1, config_roots
config_root = config_roots[0]
manifest_path = config_root / f"seed_{SEED}" / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["status"] == "complete"
assert manifest["method"] == "DEKM_F_PAPER_TEXT"
assert manifest["protocol_id"] == PROTOCOL
assert manifest["config"]["kernel_initializer"] == "glorot_uniform"
assert manifest["config"]["eigen_solver"] == "symmetric_eigh"
assert manifest["config"]["refinement_batch_size"] == manifest["n_samples"]
assert manifest["config"]["assignment_change_tolerance"] == 0.001
assert manifest["config"]["input_normalization"] == (
    "paper_mean_squared_norm_one"
)
assert manifest["source_sha256"] == "5d2524e9798710e5733755b44735b1d5519ab898ad1aa32d7c9b6491c6637bb4"

result = {
    "dataset": "RCV1",
    "seed": SEED,
    "protocol_id": PROTOCOL,
    "source_sha256": manifest["source_sha256"],
    "acc": manifest["metrics"]["acc"],
    "nmi": manifest["metrics"]["nmi"],
    "fit_time_seconds": manifest["fit_time_seconds"],
    "paper_acc": PAPER["acc"],
    "paper_nmi": PAPER["nmi"],
    "paper_acc_delta_pp": 100 * (manifest["metrics"]["acc"] - PAPER["acc"]),
    "paper_nmi_delta_pp": 100 * (manifest["metrics"]["nmi"] - PAPER["nmi"]),
    "released_source_seed42_acc": RELEASED_SOURCE_SEED42["acc"],
    "released_source_seed42_nmi": RELEASED_SOURCE_SEED42["nmi"],
    "released_source_acc_delta_pp": 100 * (
        manifest["metrics"]["acc"] - RELEASED_SOURCE_SEED42["acc"]
    ),
    "released_source_nmi_delta_pp": 100 * (
        manifest["metrics"]["nmi"] - RELEASED_SOURCE_SEED42["nmi"]
    ),
}
pd.DataFrame([result]).to_csv(REPORT_ROOT / "diagnostic.csv", index=False)
(REPORT_ROOT / "diagnostic.json").write_text(
    json.dumps(result, indent=2), encoding="utf-8"
)
display(pd.DataFrame([result]))


In [ ]:
import hashlib
import shutil

zip_path = Path(shutil.make_archive(
    "/kaggle/working/dekm-f-paper-text-rcv1-seed42",
    "zip",
    REPORT_ROOT,
))
digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
sha_path = zip_path.with_suffix(".zip.sha256")
sha_path.write_text(f"{digest}  {zip_path.name}\n", encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)
print("SHA file:", sha_path)
